In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_parquet("/Users/eric/Desktop/cali_air_quality/data/pm25_complete_model_ready.parquet")

### proximity score (to eliminate NaN in fire dist)

In [4]:
eps = 1e-4

dist_cols = [c for c in df.columns if 'dist' in c and ('fire' in c or 'smoke' in c)]

for col in dist_cols:
    new_col = col.replace('dist', 'prox')
    # Apply formula: 1 / (distance + epsilon)
    df[new_col] = 1.0 / (df[col] + eps)
    # Fill NaNs with 0.0 (No event = 0 proximity)
    df[new_col] = df[new_col].fillna(0.0)
    
if 'met_station_dist_km' in df.columns:
    df['met_prox'] = 1.0 / (df['met_station_dist_km'] + eps)
    df['met_prox'] = df['met_prox'].fillna(0.0)

In [5]:
keep_cols = [
    'site_id', 'datetime', 
    'pm_filled', 'qc_weight',          # Target & Weight
    'wind_speed', 'wind_dir',          # Weather
    'met_prox'                         # Weather Confidence
]

prox_cols = [c for c in df.columns if 'prox' in c and ('fire' in c or 'smoke' in c)]
keep_cols.extend(prox_cols)

df_ready = df[keep_cols].copy()

# Sort for time series (Critical for TCN)
df_ready = df_ready.sort_values(['site_id', 'datetime']).reset_index(drop=True)

In [ ]:
print("Final Data Shape:", df_ready.shape)
print("Missing Values:\n", df_ready.isna().sum().sum()) 
print("\nColumns:", df_ready.columns.tolist())

Final Data Shape: (635159, 13)
Missing Values:
 31776

Columns: ['site_id', 'datetime', 'pm_filled', 'qc_weight', 'wind_speed', 'wind_dir', 'met_prox', 'smoke_prox_1', 'smoke_prox_2', 'smoke_prox_3', 'fire_prox_1', 'fire_prox_2', 'fire_prox_3']


In [ ]:
print("Missing values by column AFTER fixing:")
print(df_ready.isna().sum())

Missing values by column BEFORE fixing:
site_id             0
datetime            0
pm_filled       31776
qc_weight           0
wind_speed          0
wind_dir            0
met_prox            0
smoke_prox_1        0
smoke_prox_2        0
smoke_prox_3        0
fire_prox_1         0
fire_prox_2         0
fire_prox_3         0
dtype: int64


In [9]:
# Check missing values per site
missing_by_site = df_ready[df_ready['pm_filled'].isna()].groupby('site_id').size()
print(missing_by_site)

site_id
06-001-0013-3    6911
06-007-0008-3    1464
06-013-0002-3    1829
06-019-0011-3    1300
06-029-0010-3    1101
06-037-1103-3     300
06-037-4008-3    5228
06-039-2010-3     534
06-059-0007-3    1126
06-061-0003-1    1627
06-065-8001-3     587
06-075-0005-3    1823
06-077-2010-3    1856
06-085-0002-3    3220
06-099-0006-3    1222
06-103-0007-3    1648
dtype: int64


In [11]:
# Calculate total rows per site
site_counts = df_ready.groupby('site_id').size()

# Get missing counts (you already have this, but recalculating to align)
missing_counts = df_ready[df_ready['pm_filled'].isna()].groupby('site_id').size()

# Combine into a DataFrame
missing_stats = pd.DataFrame({'Total_Rows': site_counts, 'Missing_Rows': missing_counts})
missing_stats['Pct_Missing'] = (missing_stats['Missing_Rows'] / missing_stats['Total_Rows']) * 100

# Sort by percentage
print(missing_stats.sort_values('Pct_Missing', ascending=False))

               Total_Rows  Missing_Rows  Pct_Missing
site_id                                             
06-001-0013-3       32875          6911    21.022053
06-037-4008-3       43848          5228    11.923007
06-085-0002-3       43113          3220     7.468745
06-029-0010-3       25824          1101     4.263476
06-077-2010-3       43843          1856     4.233287
06-013-0002-3       43845          1829     4.171513
06-075-0005-3       43847          1823     4.157639
06-103-0007-3       43784          1648     3.763932
06-061-0003-1       43847          1627     3.710630
06-007-0008-3       43821          1464     3.340864
06-019-0011-3       43848          1300     2.964787
06-099-0006-3       43824          1222     2.788426
06-059-0007-3       43128          1126     2.610833
06-065-8001-3       26304           587     2.231600
06-039-2010-3       43848           534     1.217843
06-037-1103-3       25560           300     1.173709


In [14]:
# 1. DROP the site with 21% missing data (Too damaged to use)
df_clean = df_ready[df_ready['site_id'] != '06-001-0013-3'].copy()

# 2. SORT to ensure time-based interpolation works
df_clean = df_clean.sort_values(by=['site_id', 'datetime'])
df_clean = df_clean.set_index('datetime')

# 3. INTERPOLATE with a limit
# We use a limit of 8 hours. EPA monitors rarely go offline for >8 hours 
# unless something is actually broken.
limit_hours = 8 
df_clean['pm_filled'] = df_clean.groupby('site_id')['pm_filled'].transform(
    lambda x: x.interpolate(method='time', limit=limit_hours)
)

# 4. HANDLE REMAINING GAPS
# If gaps > 8 hours exist, we drop those specific rows.
before_drop = len(df_clean)
df_clean = df_clean.dropna(subset=['pm_filled'])
lost_rows = before_drop - len(df_clean)

df_clean = df_clean.reset_index()

print(f"Dropped {lost_rows} rows that were part of large gaps (> {limit_hours} hours).")
print(f"Final shape: {df_clean.shape}")

Dropped 20150 rows that were part of large gaps (> 8 hours).
Final shape: (582134, 13)


In [16]:
df_clean.to_parquet("pm25_tcn.parquet")